In [1]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

In [2]:
from ingest import load_faq_data

documents = load_faq_data()

In [6]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/25 [00:00<?, ?it/s]

1208

In [8]:
vs_index.fit(vectors, documents)

In [9]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [10]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [11]:
results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [12]:
vs_index.close()

In [13]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer('all-MiniLM-L6-v2')

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [14]:
query_vector = model.encode('How do I run Kafka?')
results = vs_index.search(query_vector, num_results=5)

In [20]:
from rag_helper import RAGVector


In [23]:
from openai import OpenAI
openai_client=OpenAI()
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [24]:
vector_assistant.rag('the program has already begun, can I still sign up?')

'Yes, you can still sign up. If you want a certificate, make sure you submit your project while submissions are still open.'

In [25]:
vs_index.close()